In [1]:
# 1. Import libraries and helper functions

import pandas as pd
from pathlib import Path
import os

def normalize_text(series):
    return series.astype(str).str.strip().str.title()

In [2]:
# 2. Path configuration

BASE_DIR = Path(os.environ.get("MASKING_DATA_PATH", "../data"))
RAW_DIR = BASE_DIR / "raw" / "us"
PROCESSED_DIR = BASE_DIR / "processed" / "us"

In [3]:
# 3. Load dataset

df = pd.read_csv(RAW_DIR / "customers_raw.csv").dropna(how='all')

In [4]:
# 4. Filter dataset columns

filtered_columns = ['CUSTOMER ID', 'CUST_NAME', 'CUST_GROUP', 'REPRESENTATIVE', 'MANAGER']

df = df[filtered_columns].copy()

In [5]:
# 5. Rename columns

df = df.rename(columns={
    'CUSTOMER ID': 'cust_id',
    'CUST_NAME': 'cust_name',
    'CUST_GROUP': 'cust_group',
    'REPRESENTATIVE': 'rep_name',
    'MANAGER': 'rep_manager',
})

In [6]:
# 6. Validate before changing anything

assert df['cust_id'].duplicated().sum() == 0, "cust_id duplicated!"
assert df['cust_name'].duplicated().sum() == 0, "cust_name duplicated!"

In [7]:
# 7. Convert data types

df['cust_id'] = df['cust_id'].astype(str)
df['cust_name'] = normalize_text(df['cust_name'])
df['cust_group'] = normalize_text(df['cust_group'])
df['rep_name'] = normalize_text(df['rep_name']).astype('category')
df['rep_manager'] = normalize_text(df['rep_manager']).astype('category')

In [8]:
# 8. Final checks

assert df['cust_id'].duplicated().sum() == 0, "cust_id duplicated"
assert df.isna().sum().sum() == 0, "Null values detected!"

In [9]:
# 9. Cross-validate rep_name against reps_dim

reps_dim = pd.read_csv(PROCESSED_DIR / "dim_reps.csv")
valid_reps = reps_dim['rep_name'].unique()

orphans = df[~df['rep_name'].isin(valid_reps)]
assert len(orphans) == 0, f"{len(orphans)} customer with invalid representative found!"

In [10]:
# 9. Export cleaned dataset for BI 

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

df.to_csv(PROCESSED_DIR / "dim_cust.csv", index=False)

print(f"Success! {len(df)} cust exported.")

Success! 2666 cust exported.
